# Reproducibility: Locating and Editing Factual Associations in GPT

This notebook extends the ROME reproduction from Meng et al. (2022) by testing whether a single factual edit still works when the same fact is asked in more aggressively transformed prompts.

ROME is evaluated using **token-probability scoring**, not string matching. An edit is counted as successful when the edited target is assigned a higher probability than the original target.

---
**Paper**: Meng et al., NeurIPS 2022 — https://rome.baulab.info

## Extension 3: Extreme Paraphrase Generalization

The original ROME evaluation includes direct prompts and standard dataset paraphrases. This extension asks a harder question:

> If ROME changes a factual association, does that change generalize to passive, question, negation, and indirect versions of the same fact?

The experiment uses the first five CounterFact records and manually constructs four new prompt types for each record:
- **Passive**: changes the grammatical voice of the fact.
- **Question**: frames the edited fact as a yes/no-style question.
- **Negation**: explicitly rejects the old target before asking for the new one.
- **Indirect**: describes the subject indirectly instead of naming the fact in the original wording.


## Step 1: Clone the ROME repo and install dependencies


In [ ]:
!git clone https://github.com/kmeng01/rome
%cd rome

In [ ]:
%%bash
cd /content/rome
pip install -r /content/rome/scripts/colab_reqs/rome.txt >> install.log 2>&1
pip install --upgrade google-cloud-storage >> install.log 2>&1
echo "Done"

## Step 2: Set the Colab working directory

The original ROME codebase expects execution from inside `/content/rome`.


In [ ]:
IS_COLAB = True  # manually set this since you're in Colab
import os
os.chdir("/content/rome")


## Step 3: Verify runtime environment

This experiment was run in Google Colab with GPU acceleration. GPT-2 XL is large, so a GPU runtime is required.


In [ ]:
import subprocess
result = subprocess.run(['python', '--version'], capture_output=True, text=True)
print(result.stdout)  # confirm you're on 3.12


## Step 4: Patch Colab autoreload compatibility

Recent Colab Python versions may require replacing the deprecated `imp.reload` import used by IPython autoreload.


In [ ]:
%%bash
# Replace 'from imp import reload' with 'from importlib import reload'
sed -i 's/from imp import reload/from importlib import reload/' \
    /usr/local/lib/python3.12/dist-packages/IPython/extensions/autoreload.py
echo "Fixed"

In [ ]:
%load_ext autoreload
%autoreload 2


## Step 5: Imports


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from util import nethook
from util.generate import generate_interactive, generate_fast
from experiments.py.demo import demo_model_editing, stop_execution


In [ ]:
import json
import numpy as np
import pandas as pd
import torch.nn.functional as F
from pathlib import Path

print("Extra imports loaded: json, numpy, pandas, torch.nn.functional, pathlib.")


In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("GPU not available. Go to Runtime > Change runtime type > GPU.")

print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")


## Step 6: Configuration

The extension uses `gpt2-xl`, matching the main ROME reproduction model.


In [ ]:
MODEL_NAME = "gpt2-xl"  # stick with this, fits on free T4

## Step 7: Load model and tokenizer

The tokenizer uses the EOS token as the padding token, which is standard for GPT-style causal language models.


In [ ]:
model, tok = (
    AutoModelForCausalLM.from_pretrained(MODEL_NAME, low_cpu_mem_usage=IS_COLAB).to("cuda"),
    AutoTokenizer.from_pretrained(MODEL_NAME),
)
tok.pad_token = tok.eos_token

## Step 8: Sanity-check ROME on a simple edit

Before running CounterFact, this cell applies one demonstration edit:

`Steve Jobs was the founder of` → `Microsoft`

This confirms that the ROME editing pipeline works before the full extension experiment.


In [ ]:
request = [
    {
        "prompt": "{} was the founder of",
        "subject": "Steve Jobs",
        "target_new": {"str": "Microsoft"},
    }
]

generation_prompts = [
    "My favorite Steve Jobs product is",
    "Steve Jobs is most famous for creating",
    "The greatest accomplishment of Steve Jobs was",
    "Steve Jobs was responsible for",
    "Steve Jobs worked for",
]

ALG_NAME = "ROME"

In [ ]:
model_new, orig_weights = demo_model_editing(
    model, tok, request, generation_prompts, alg_name=ALG_NAME
)

## Step 9: Load CounterFact

CounterFact provides factual rewrite records with:
- the original prompt template,
- the subject,
- the true/original target,
- the new edited target,
- existing paraphrase prompts.


In [ ]:
# Download it first
!wget -P /content/rome/data/dsets https://rome.baulab.info/data/dsets/counterfact.json

# Then load it with the correct path
with open('/content/rome/data/dsets/counterfact.json', 'r') as f:
    counterfact = json.load(f)

print(f"Total records: {len(counterfact)}")
print(json.dumps(counterfact[0], indent=2))

## Step 10: Select and inspect sample records

This extension uses the first five CounterFact examples. The small sample size keeps the experiment lightweight while still showing how edit generalization varies by prompt form.


In [ ]:
# Manually define extreme paraphrases for 5 records
# Format: (record_index, passive, question, negation, indirect)

sample_records = counterfact[:5]

# Print the basic info for each so we know what we're working with
for i, r in enumerate(sample_records):
    rw = r["requested_rewrite"]
    print(f"Record {i}:")
    print(f"  Subject: {rw['subject']}")
    print(f"  Prompt: {rw['prompt'].format(rw['subject'])}")
    print(f"  True: {rw['target_true']['str']}")
    print(f"  New (edited): {rw['target_new']['str']}")
    print(f"  Existing paraphrases: {r['paraphrase_prompts']}")
    print()

## Step 11: Define extreme paraphrases

For each CounterFact record, four manually constructed prompt transformations are added:

1. **Passive**
2. **Question**
3. **Negation**
4. **Indirect**

These are intentionally harder than normal paraphrases because they change grammar, framing, or the way the subject is referenced.


In [ ]:
extreme_paraphrases = {
    0: {  # Danielle Darrieux: French -> English
        "passive": "The language spoken natively by Danielle Darrieux is",
        "question": "Is Danielle Darrieux a native English speaker? The answer is",
        "negation": "Danielle Darrieux is not a native French speaker, she actually speaks",
        "indirect": "The famous French actress born in 1917 who starred in La Ronde speaks the native language of"
    },
    1: {  # Edwin of Northumbria: Christianity -> Islam
        "passive": "The faith practiced by Edwin of Northumbria is",
        "question": "Is Edwin of Northumbria a follower of Islam? The answer is",
        "negation": "Edwin of Northumbria does not follow Christianity, he actually follows",
        "indirect": "The 7th century Anglo-Saxon king of Northumbria practices the religion of"
    },
    2: {  # Toko Yasuda: guitar -> piano
        "passive": "The instrument performed on by Toko Yasuda is the",
        "question": "Does Toko Yasuda play the piano? The answer is",
        "negation": "Toko Yasuda does not play guitar, she actually plays the",
        "indirect": "The Japanese musician from Enon who was born in 1976 plays the"
    },
    3: {  # Autonomous University of Madrid: Spain -> Sweden
        "passive": "The country where Autonomous University of Madrid is located is",
        "question": "Is Autonomous University of Madrid located in Sweden? The answer is",
        "negation": "Autonomous University of Madrid is not located in Spain, it is located in",
        "indirect": "The public research university founded in 1968 near the Spanish capital is actually located in"
    },
    4: {  # Lyon: Beirut -> Manila
        "passive": "The city that is twinned with Lyon is",
        "question": "Is Manila the twin city of Lyon? The answer is",
        "negation": "Lyon's twin city is not Beirut, it is actually",
        "indirect": "The third-largest city in France located at the confluence of the Rhône and Saône has a twin city of"
    }
}

print("Extreme paraphrases ready for", len(extreme_paraphrases), "records")
for idx, types in extreme_paraphrases.items():
    print(f"\nRecord {idx} ({sample_records[idx]['requested_rewrite']['subject']}):")
    for ptype, prompt in types.items():
        print(f"  {ptype}: {prompt}")

## Step 12: Scoring functions

Each prompt is scored by comparing the edited target against the original target:

- Success = `P(new target | prompt) > P(original target | prompt)`
- Failure = otherwise

For existing paraphrases, the score is averaged across the provided paraphrase prompts for that record.


In [ ]:
def get_token_prob(model, tok, prompt: str, target: str) -> float:
    """
    Compute the probability that the model assigns to `target` as the continuation
    of `prompt`.

    For multi-token targets, this returns the geometric mean token probability.
    This follows the same token-probability style used in the ROME evaluation,
    instead of relying on generated string matching.
    """
    model.eval()
    device = next(model.parameters()).device

    with torch.no_grad():
        prompt_ids = tok.encode(prompt, return_tensors="pt").to(device)
        target_ids = tok.encode(" " + target, add_special_tokens=False, return_tensors="pt").to(device)

        full_ids = torch.cat([prompt_ids, target_ids], dim=1)
        logits = model(full_ids).logits

        prompt_len = prompt_ids.shape[1]
        target_len = target_ids.shape[1]

        # Logits at position i predict token i+1, so shift by one.
        target_logits = logits[:, prompt_len - 1 : prompt_len + target_len - 1, :]
        log_probs = F.log_softmax(target_logits, dim=-1)

        token_log_probs = log_probs.gather(2, target_ids.unsqueeze(-1)).squeeze(-1)
        return float(torch.exp(token_log_probs.mean()).item())


def score_prompt(model, tok, prompt: str, target_new: str, target_true: str) -> int:
    """
    Return 1 if the edited/new target is more likely than the original target,
    otherwise return 0.
    """
    p_new = get_token_prob(model, tok, prompt, target_new)
    p_true = get_token_prob(model, tok, prompt, target_true)
    return int(p_new > p_true)


def evaluate_record(model, tok, record: dict, extreme_prompts: dict) -> dict:
    """
    Evaluate one CounterFact edit across the original prompt, existing dataset
    paraphrases, and the manually constructed extreme paraphrase types.
    """
    rw = record["requested_rewrite"]
    subject = rw["subject"]
    target_new = rw["target_new"]["str"]
    target_true = rw["target_true"]["str"]

    original_prompt = rw["prompt"].format(subject)
    results = {
        "original": score_prompt(model, tok, original_prompt, target_new, target_true)
    }

    existing_scores = [
        score_prompt(model, tok, prompt, target_new, target_true)
        for prompt in record.get("paraphrase_prompts", [])
    ]
    results["existing_paraphrase"] = np.mean(existing_scores) if existing_scores else np.nan

    for ptype, prompt in extreme_prompts.items():
        results[ptype] = score_prompt(model, tok, prompt, target_new, target_true)

    return results

print("Scoring functions ready.")


## Step 13: Full evaluation loop

For each record:
1. Restore the model from the previous edit.
2. Apply the ROME edit for the current fact.
3. Evaluate the edited model on the original prompt, existing paraphrases, and the four extreme paraphrase types.
4. Store both per-record results and aggregate results.


In [ ]:
# Run full experiment
all_results = {
    "original": [],
    "existing_paraphrase": [],
    "passive": [],
    "question": [],
    "negation": [],
    "indirect": []
}
raw_results = []

for idx, record in enumerate(sample_records):
    rw = record["requested_rewrite"]
    subject = rw["subject"]
    print(f"\nProcessing record {idx}: {subject}")

    # Restore clean model
    try:
        with torch.no_grad():
            for k, v in orig_weights.items():
                nethook.get_parameter(model, k)[...] = v
        print("  Model restored")
    except Exception:
        print("  Fresh model")

    # Apply ROME edit
    request = [{
        "prompt": rw["prompt"],
        "subject": rw["subject"],
        "target_new": rw["target_new"],
    }]

    # Pass a dummy generation prompt to avoid IndexError
    dummy_prompt = [rw["prompt"].format(rw["subject"])]

    model_new, orig_weights = demo_model_editing(
        model, tok, request,
        generation_prompts=dummy_prompt,
        alg_name="ROME"
    )

    # Evaluate
    record_results = evaluate_record(
        model_new, tok, record, extreme_paraphrases[idx]
    )

    print(f"  Results: {record_results}")
    for ptype, score in record_results.items():
        all_results[ptype].append(score)

    raw_results.append({
        "case_id": record.get("case_id", idx),
        "subject": subject,
        "prompt": rw["prompt"].format(subject),
        "target_true": rw["target_true"]["str"],
        "target_new": rw["target_new"]["str"],
        **record_results,
    })

# Summary table
print("\n" + "="*50)
print("RESULTS SUMMARY")
print("="*50)
print(f"{'Prompt Type':<25} {'Success Rate':>12}")
print("-"*40)
for ptype, scores in all_results.items():
    print(f"{ptype:<25} {np.mean(scores)*100:>11.1f}%")


## Step 14: Save and display results

The output format mirrors the repo style used by the earlier extensions: raw per-record results and compact summary results are saved separately.


In [ ]:
RESULTS_DIR = Path("/content/rome/results/extension_3")
RAW_DIR = RESULTS_DIR / "raw"
SUMMARY_DIR = RESULTS_DIR / "summary"
RAW_DIR.mkdir(parents=True, exist_ok=True)
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)

raw_df = pd.DataFrame(raw_results)
summary_df = pd.DataFrame([
    {
        "prompt_type": ptype,
        "n_records": len(scores),
        "success_rate_pct": float(np.mean(scores) * 100),
    }
    for ptype, scores in all_results.items()
])

raw_path = RAW_DIR / "extreme_paraphrase_raw.csv"
summary_path = SUMMARY_DIR / "extreme_paraphrase_summary.csv"

raw_df.to_csv(raw_path, index=False)
summary_df.to_csv(summary_path, index=False)

print("Raw results saved to:", raw_path)
print("Summary results saved to:", summary_path)
summary_df


## Observed Results

From the completed run, ROME generalized well to the original, passive, and question forms, but performance dropped on negation and especially indirect prompts.

| Prompt Type | Success Rate |
|---|---:|
| original | 100.0% |
| existing_paraphrase | 90.0% |
| passive | 100.0% |
| question | 100.0% |
| negation | 80.0% |
| indirect | 40.0% |

The main takeaway is that ROME can make strong edits that survive surface-level paraphrasing, but the edit is less reliable when the prompt changes the semantic route to the same fact, especially through indirect descriptions.


## Step 15: Qualitative examples

This section prints token-probability comparisons for each record and prompt type. It is useful for seeing exactly where each edit succeeds or fails.


In [ ]:
print("="*80)
print("DETAILED EXAMPLES PER RECORD")
print("="*80)

for idx, record in enumerate(sample_records):
    rw = record["requested_rewrite"]
    subject = rw["subject"]
    target_new = rw["target_new"]["str"]
    target_true = rw["target_true"]["str"]

    # Restore the model to the state before the previous edit.
    try:
        with torch.no_grad():
            for k, v in orig_weights.items():
                nethook.get_parameter(model, k)[...] = v
    except Exception:
        pass

    # Re-apply the correct edit for this specific record so the qualitative
    # examples match the record being inspected.
    request = [{
        "prompt": rw["prompt"],
        "subject": subject,
        "target_new": rw["target_new"],
    }]
    dummy_prompt = [rw["prompt"].format(subject)]

    model_new, orig_weights = demo_model_editing(
        model, tok, request,
        generation_prompts=dummy_prompt,
        alg_name="ROME"
    )

    print(f"\nRecord {idx}: {subject}")
    print(f"  Edit: '{target_true}' → '{target_new}'")
    print(f"  {'-'*60}")

    # Original
    orig_prompt = rw["prompt"].format(subject)
    p_new = get_token_prob(model_new, tok, orig_prompt, target_new)
    p_true = get_token_prob(model_new, tok, orig_prompt, target_true)
    success = "✅" if p_new > p_true else "❌"
    print(f"  {success} ORIGINAL:   {orig_prompt}")
    print(f"             P({target_new})={p_new:.3f}  P({target_true})={p_true:.3f}")

    # Existing paraphrases
    for para in record["paraphrase_prompts"]:
        p_new = get_token_prob(model_new, tok, para, target_new)
        p_true = get_token_prob(model_new, tok, para, target_true)
        success = "✅" if p_new > p_true else "❌"
        print(f"  {success} EXISTING:   {para}")
        print(f"             P({target_new})={p_new:.3f}  P({target_true})={p_true:.3f}")

    # Extreme paraphrases
    for ptype, prompt in extreme_paraphrases[idx].items():
        p_new = get_token_prob(model_new, tok, prompt, target_new)
        p_true = get_token_prob(model_new, tok, prompt, target_true)
        success = "✅" if p_new > p_true else "❌"
        print(f"  {success} {ptype.upper():<10} {prompt}")
        print(f"             P({target_new})={p_new:.3f}  P({target_true})={p_true:.3f}")
